# LLM Engine & Dynamic RAG Testing Notebook

This notebook completes the RAG pipeline by adding:
1. **Local LLM Engine**: Connects to local Ollama (`llama3:latest`) via API.
2. **Conversation Manager**: Tracks the last 5 queries and answers.
3. **Sentiment Analysis**: Uses `textblob` to detect if the user is confused or understanding well.
4. **Dynamic Retrieval**: Adjusts the number of chunks retrieved (`k`) and the detail of the LLM response based on the user's sentiment.

## Cell 1 - Install Dependencies

In [ ]:
!pip install textblob requests chromadb sentence-transformers pandas
!python -m textblob.download_corpora

## Cell 2 - Imports

In [ ]:
import requests
import json
import chromadb
from sentence_transformers import SentenceTransformer
from textblob import TextBlob

print("All imports successful!")

## Cell 3 - Load Embedding Model & Vector Database

We will use the ChromaDB collection we built previously. It natively supports metadata filtering.

In [ ]:
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading ChromaDB...")
CHROMA_DB_PATH = './chroma_db'
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_collection(name="rag_collection")

print(f"Database loaded with {collection.count()} vectors.")

## Cell 4 - Conversation & Sentiment Manager

Tracks history and calculates average sentiment over recent queries to detect confusion.

In [ ]:
class ConversationManager:
    def __init__(self, max_history=5):
        self.max_history = max_history
        self.history = []  # List of dicts: {'query': ..., 'answer': ..., 'sentiment': ...}

    def analyze_sentiment(self, text):
        """Returns polarity score between -1.0 (very negative/confused) and 1.0 (very positive/clear)"""
        blob = TextBlob(text)
        return blob.sentiment.polarity

    def add_turn(self, query, answer):
        sentiment = self.analyze_sentiment(query)
        self.history.append({
            'query': query,
            'answer': answer,
            'sentiment': sentiment
        })
        # Keep only the last 'max_history' turns
        if len(self.history) > self.max_history:
            self.history.pop(0)

    def get_average_sentiment(self, current_query=None):
        """Calculates average sentiment of history + current query."""
        scores = [turn['sentiment'] for turn in self.history]
        if current_query:
            scores.append(self.analyze_sentiment(current_query))
            
        if not scores:
            return 0.0
        return sum(scores) / len(scores)
    
    def get_formatted_history(self):
        """Formats history for the LLM prompt."""
        if not self.history:
            return "No previous conversation history."
        
        formatted = ""
        for i, turn in enumerate(self.history):
            formatted += f"User (Turn {i+1}): {turn['query']}\n"
            formatted += f"AI (Turn {i+1}): {turn['answer']}\n\n"
        return formatted.strip()

print("[OK] ConversationManager defined!")

## Cell 5 - Local Ollama Engine

Connects to the local Ollama instance running `llama3:latest`.

In [ ]:
class LocalLLMEngine:
    def __init__(self, model_name="llama3:latest", base_url="http://localhost:11434"):
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"
        
    def check_connection(self):
        """Check if Ollama is running."""
        try:
            response = requests.get(self.base_url)
            return response.status_code == 200
        except requests.exceptions.ConnectionError:
            return False

    def generate(self, prompt, system_instruction=""):
        """Calls the Ollama API with the prompt and system instructions."""
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "system": system_instruction,
            "stream": False
        }
        
        try:
            response = requests.post(self.api_url, json=payload)
            response.raise_for_status()
            return response.json().get("response", "")
        except requests.exceptions.RequestException as e:
            return f"[ERROR] LLM Generation failed: {e}"

print("[OK] LocalLLMEngine defined!")

## Cell 6 - Dynamic RAG Pipeline

Ties together Retrieval, Sentiment Analysis, and LLM Generation.

In [ ]:
class DynamicRAGPipeline:
    def __init__(self, collection, embedding_model, llm_engine, base_k=5):
        self.collection = collection
        self.embedding_model = embedding_model
        self.llm_engine = llm_engine
        self.base_k = base_k
        self.convo_manager = ConversationManager(max_history=5)
        
    def _retrieve_chunks(self, query, k, filters=None):
        """Helper to fetch chunks from ChromaDB."""
        query_embedding = self.embedding_model.encode([query]).tolist()
        
        where_filter = None
        if filters:
            conditions = [{key: {"$eq": value}} for key, value in filters.items()]
            where_filter = conditions[0] if len(conditions) == 1 else {"$and": conditions}
            
        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=k,
            where=where_filter
        )
        
        if not results['documents'] or not results['documents'][0]:
            return []
        return results['documents'][0]

    def chat(self, query, filters=None):
        """Main chat endpoint for the user."""
        # 1. Analyze sentiment
        avg_sentiment = self.convo_manager.get_average_sentiment(query)
        
        # 2. Dynamic Rules based on Sentiment
        # If sentiment is negative (confused/frustrated) -> increase K, write detailed response
        # If sentiment is highly positive -> decrease K, write concise/precise response
        # Otherwise -> normal K, standard response
        
        if avg_sentiment < -0.1:  # Confused / Negative
            k = self.base_k + 3
            system_instruction = (
                "You are a highly patient and detailed tutor. The user seems confused or is struggling "
                "to understand. Provide a very detailed, step-by-step explanation. Break down complex "
                "concepts simply. Use the provided context to answer accurately."
            )
            state_msg = "User is confused -> Increasing K, providing detailed response."
            
        elif avg_sentiment > 0.3: # Clear / Understanding
            k = max(2, self.base_k - 2)
            system_instruction = (
                "You are an expert tutor. The user is understanding the material well. "
                "Provide highly precise, accurate, and concise answers without unnecessary fluff. "
                "Use the provided context to answer."
            )
            state_msg = "User understands -> Decreasing K, providing precise response."
            
        else: # Neutral
            k = self.base_k
            system_instruction = (
                "You are a helpful tutor. Use the provided context to answer the user's question clearly. "
                "If the context doesn't contain the answer, say you don't know based on the book."
            )
            state_msg = "User is neutral -> Standard K, standard response."

        print(f"[LOG] Avg Sentiment: {avg_sentiment:.2f} | {state_msg} (k={k})")
        
        # 3. Retrieve Context
        chunks = self._retrieve_chunks(query, k, filters)
        context_str = "\n\n---\n\n".join(chunks)
        
        # 4. Format Prompt
        history_str = self.convo_manager.get_formatted_history()
        
        prompt = f"""**Conversation History (Last 5 turns):**
{history_str}

**Retrieved Knowledge Chunks:**
{context_str}

**Current User Query:**
{query}

**Your Answer:**
"""
        
        # 5. Generate Response via LLM
        print("[LOG] Generating response via Ollama...")
        response = self.llm_engine.generate(prompt, system_instruction)
        
        # 6. Update History
        self.convo_manager.add_turn(query, response)
        
        return response

print("[OK] DynamicRAGPipeline defined!")

## Cell 7 - Initialize the System

In [ ]:
llm = LocalLLMEngine(model_name="llama3:latest")
rag = DynamicRAGPipeline(collection, model, llm, base_k=5)

if llm.check_connection():
    print("\n[OK] Connected to local Ollama API!")
else:
    print("\n[WARN] Could not connect to Ollama. Make sure you run 'ollama serve' and have the 'llama3:latest' model.")

## Cell 8 - Interactive Chat Test

Run this cell to test the pipeline. Watch how it reacts when you act confused vs when you understand!

In [ ]:
def simulate_conversation():
    print("Starting simulation... (Make sure Ollama is running)\n")
    
    queries = [
        "What is Newton's second law of motion?",
        "I don't get it, this is very confusing. Can you explain it again simply?",
        "Oh, I understand now! Excellent explanation. What is the formula?"
    ]
    
    for i, q in enumerate(queries):
        print(f"\n{'='*60}")
        print(f"USER (Turn {i+1}): {q}")
        print(f"{'='*60}")
        
        ans = rag.chat(q, filters={"subject": "Physics"})  # Filtering to Physics only
        
        print(f"\nAI RESPONSE:\n{ans}")

# Uncomment the line below to run the simulation if Ollama is running:
simulate_conversation()